# Hyperparameter Search Notebook

In [58]:
import os
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline as ModelPipeline
from xgboost import XGBClassifier

sns.set_theme(style='whitegrid', font_scale=0.9)
pd.set_option('display.max_columns', 80)

RANDOM_STATE = 42
PHASE1_RERUN = True
PHASE2_RERUN = False
FIGURE_DPI = 160
CPU_COUNT = os.cpu_count() or 1
SEARCH_N_JOBS = max(1, min(8, CPU_COUNT))
MODEL_FIT_N_JOBS = 1
SEARCH_PRE_DISPATCH = SEARCH_N_JOBS
SEARCH_VERBOSE = 2

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.pipeline import build_pipeline, load_feature_frame, random_split

results_dir = project_root / 'results'
results_dir.mkdir(parents=True, exist_ok=True)
data_path = project_root / 'data' / 'processed' / 'matches.parquet'
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
phase1_models = ('LR', 'RF', 'XGB')

print(f'project_root={project_root}')
print(f'data_path={data_path}')
print(f'results_dir={results_dir}')
print(f'phase1_rerun={PHASE1_RERUN}')
print(f'phase2_rerun={PHASE2_RERUN}')
print(f'cpu_count={CPU_COUNT}')
print(f'search_n_jobs={SEARCH_N_JOBS}')
print(f'model_fit_n_jobs={MODEL_FIT_N_JOBS}')
print(f'search_verbose={SEARCH_VERBOSE}')

project_root=/mnt/ncsudrive/x/xzhou38/CSC522-S26-proj-deadlock-match-forecast
data_path=/mnt/ncsudrive/x/xzhou38/CSC522-S26-proj-deadlock-match-forecast/data/processed/matches.parquet
results_dir=/mnt/ncsudrive/x/xzhou38/CSC522-S26-proj-deadlock-match-forecast/results
phase1_rerun=True
phase2_rerun=False
cpu_count=16
search_n_jobs=8
model_fit_n_jobs=1
search_verbose=2


In [14]:
df_feat, y, df_meta = load_feature_frame(data_path, feature_set='clean')
splits = random_split(len(df_feat), y, test_size=0.20, val_size=0.125, seed=RANDOM_STATE)

X_train_raw = df_feat.iloc[splits['train']].copy()
X_val_raw = df_feat.iloc[splits['val']].copy()
X_test_raw = df_feat.iloc[splits['test']].copy()
y_train = y.iloc[splits['train']].copy()
y_val = y.iloc[splits['val']].copy()
y_test = y.iloc[splits['test']].copy()

split_summary = pd.DataFrame([
    {
        'split': split_name,
        'rows': len(indices),
        'label_rate': y.iloc[indices].mean(),
    }
    for split_name, indices in splits.items()
])

feature_summary = pd.DataFrame([
    {'surface': 'clean_raw', 'rows': df_feat.shape[0], 'cols': df_feat.shape[1]},
    {'surface': 'train_raw', 'rows': X_train_raw.shape[0], 'cols': X_train_raw.shape[1]},
    {'surface': 'val_raw', 'rows': X_val_raw.shape[0], 'cols': X_val_raw.shape[1]},
    {'surface': 'test_raw', 'rows': X_test_raw.shape[0], 'cols': X_test_raw.shape[1]},
])

usage_summary = pd.DataFrame([
    {'split': 'train', 'used_for': 'cross-validation and hyperparameter search'},
    {'split': 'val', 'used_for': 'post-selection checks only'},
    {'split': 'test', 'used_for': 'final report only'},
])

display(feature_summary)
display(split_summary.round(4))
display(usage_summary)

,surface,rows,cols
0,clean_raw,100039,112
1,train_raw,70027,112
2,val_raw,10004,112
3,test_raw,20008,112


,split,rows,label_rate
0,train,70027,0.5052
1,val,10004,0.5052
2,test,20008,0.5052


,split,used_for
0,train,cross-validation and hyperparameter search
1,val,post-selection checks only
2,test,final report only


## Phase 1 Search

Run Phase 1 on the raw training fold only. Each model starts from a reasonable anchor configuration, then sweeps one hyperparameter at a time while holding the others fixed. Use broader initial grids and finer follow-up grids across multiple Phase 1 rounds until each parameter range is stable enough for Phase 2 joint search. Preprocessing is nested inside each search estimator so every CV split fits its own transformer state. Validation and test splits remain untouched during model selection.

In [59]:
phase1_csv_paths = {
    'LR': results_dir / 'hpsearch_lr_p1.csv',
    'RF': results_dir / 'hpsearch_rf_p1_round3.csv',
    'XGB': results_dir / 'hpsearch_xgb_p1.csv',
}

lr_p1_anchor = {'C': 1.0}
lr_p1_sweeps = {
    'C': [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100],
}

# Round 3 anchor: n_estimators=500 (cost-capped), max_features=0.04 (round 3 best, lower boundary confirmed).
rf_p1_anchor = {
    'n_estimators': 500,
    'max_depth': 28,
    'min_samples_leaf': 4,
    'max_features': 0.04,
}
rf_p1_confirmed_space = {
    # Round 2 did not add evidence to reopen depth; keep the compact carry-over space from the earlier scan.
    'max_depth': [16, 20, 28, None],
    # Round 2 still supports the smoother leaf sizes; keep the anti-overfitting compact space.
    'min_samples_leaf': [4, 6, 8, 12],
    # Monotonically improving across all rounds; locked to a cost-practical range for Phase 2.
    'n_estimators': [300, 500, 700],
    # Round 3 best at 0.04 (lower boundary); Phase 2 will fine-scan around this value.
    'max_features': [0.04],
}
rf_p1_sweeps = {
    # Round 2 best at 0.08 on the lower edge; rescan further downward and keep sqrt as a coarse reference.
    'max_features': [0.04, 0.06, 0.08, 0.1, 0.12, 'sqrt'],
}

xgb_p1_anchor = {
    'learning_rate': 0.1,
    'max_depth': 6,
    'n_estimators': 300,
    'subsample': 0.9,
    'colsample_bytree': 1.0,
}
xgb_p1_sweeps = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2, 0.3],
    'max_depth': [3, 4, 5, 6, 8, 10],
    'n_estimators': [100, 200, 300, 500, 700],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.8, 0.9, 1.0],
}

phase1_anchor_params = {
    'LR': lr_p1_anchor,
    'RF': rf_p1_anchor,
    'XGB': xgb_p1_anchor,
}
phase1_sweep_spaces = {
    'LR': lr_p1_sweeps,
    'RF': rf_p1_sweeps,
    'XGB': xgb_p1_sweeps,
}

def build_model_pipeline(estimator):
    return ModelPipeline([
        ('preprocess', build_pipeline()),
        ('model', estimator),
    ])


def prefix_model_params(param_space):
    return {f'model__{name}': values for name, values in param_space.items()}


def normalize_cv_results(results_df):
    rename_map = {
        column: column.replace('param_model__', 'param_')
        for column in results_df.columns
        if column.startswith('param_model__')
    }
    normalized = results_df.rename(columns=rename_map).copy()
    if 'params' in normalized.columns:
        normalized['params'] = normalized['params'].map(
            lambda params: {key.replace('model__', ''): value for key, value in params.items()}
            if isinstance(params, dict)
            else params
        )
    return normalized


def estimate_sweep_trial_count(sweep_space):
    return sum(len(candidates) for candidates in sweep_space.values())


def build_single_param_grid(anchor_params, param_name, candidates):
    grid = {f'model__{name}': [value] for name, value in anchor_params.items()}
    grid[f'model__{param_name}'] = list(candidates)
    return grid


def filter_sweep_results(results_df, param_name):
    working = results_df.copy()
    if 'sweep_param' in working.columns:
        working = working[working['sweep_param'] == param_name].copy()
    return working


def run_or_load_phase1_sweeps(model_name, estimator, anchor_params, sweep_space, X_raw, y_target, csv_path, rerun=False):
    if csv_path.exists() and not rerun:
        print(f'Loading cached Phase 1 results for {model_name} from {csv_path.name}')
        return pd.read_csv(csv_path)

    total_trials = estimate_sweep_trial_count(sweep_space)
    print(f'=== {model_name} Phase 1 ===')
    print(f'anchor_params={anchor_params}')
    print(f'output_csv={csv_path.name}')
    print(f'total_candidates={total_trials}, total_fits={total_trials * cv.get_n_splits()}')

    sweep_frames = []
    total_elapsed = 0.0
    for param_name, candidates in sweep_space.items():
        print(f'Scanning {param_name}: {list(candidates)}')
        search = GridSearchCV(
            build_model_pipeline(clone(estimator)),
            param_grid=build_single_param_grid(anchor_params, param_name, candidates),
            cv=cv,
            scoring='roc_auc',
            n_jobs=SEARCH_N_JOBS,
            pre_dispatch=SEARCH_PRE_DISPATCH,
            return_train_score=True,
            verbose=SEARCH_VERBOSE,
        )
        start = time.perf_counter()
        search.fit(X_raw, y_target)
        elapsed = time.perf_counter() - start
        total_elapsed += elapsed

        sweep_df = normalize_cv_results(pd.DataFrame(search.cv_results_))
        sweep_df['model'] = model_name
        sweep_df['sweep_param'] = param_name
        sweep_df['anchor_params'] = str(anchor_params)
        sweep_frames.append(sweep_df)

        best_row = best_trial(sweep_df)
        print(
            f'Best {param_name}: value={best_row[f"param_{param_name}"]}, '
            f'cv_auc={best_row["mean_test_score"]:.6f}, elapsed={elapsed:.1f}s'
        )

    results_df = pd.concat(sweep_frames, ignore_index=True)
    results_df = results_df.sort_values(['sweep_param', 'rank_test_score', 'mean_fit_time']).reset_index(drop=True)
    results_df.to_csv(csv_path, index=False)
    print(f'Saved {len(results_df)} trials to {csv_path.name} in {total_elapsed:.1f}s')
    return results_df


def best_trial(results_df):
    best_idx = results_df['mean_test_score'].idxmax()
    return results_df.loc[best_idx].copy()


def best_trial_per_sweep(results_df):
    sweep_best_rows = [best_trial(sweep_df) for _, sweep_df in results_df.groupby('sweep_param', sort=False)]
    return pd.DataFrame(sweep_best_rows).reset_index(drop=True)


phase1_space_summary = pd.DataFrame([
    {
        'model': 'LR',
        'search_strategy': 'one_dimensional_grid',
        'anchor_params': str(lr_p1_anchor),
        'phase1_trial_count': estimate_sweep_trial_count(lr_p1_sweeps),
        'phase1_fits': estimate_sweep_trial_count(lr_p1_sweeps) * cv.get_n_splits(),
    },
    {
        'model': 'RF',
        'search_strategy': 'one_dimensional_grid_round3',
        'anchor_params': str(rf_p1_anchor),
        'phase1_trial_count': estimate_sweep_trial_count(rf_p1_sweeps),
        'phase1_fits': estimate_sweep_trial_count(rf_p1_sweeps) * cv.get_n_splits(),
    },
    {
        'model': 'XGB',
        'search_strategy': 'one_dimensional_grid',
        'anchor_params': str(xgb_p1_anchor),
        'phase1_trial_count': estimate_sweep_trial_count(xgb_p1_sweeps),
        'phase1_fits': estimate_sweep_trial_count(xgb_p1_sweeps) * cv.get_n_splits(),
    },
])


display(phase1_space_summary)

,model,search_strategy,anchor_params,phase1_trial_count,phase1_fits
0,LR,one_dimensional_grid,{'C': 1.0},11,55
1,RF,one_dimensional_grid_round3,"{'n_estimators': 500, 'max_depth': 28, 'min_sa...",6,30
2,XGB,one_dimensional_grid,"{'learning_rate': 0.1, 'max_depth': 6, 'n_esti...",27,135


In [39]:
lr_p1_results = run_or_load_phase1_sweeps(
    'LR',
    LogisticRegression(solver='lbfgs', max_iter=1000),
    phase1_anchor_params['LR'],
    phase1_sweep_spaces['LR'],
    X_train_raw,
    y_train,
    phase1_csv_paths['LR'],
    rerun=PHASE1_RERUN,
)

lr_p1_view = lr_p1_results[
    ['sweep_param', 'param_C', 'mean_test_score', 'std_test_score', 'mean_train_score', 'std_train_score', 'rank_test_score']
] .copy()
lr_p1_view['param_C'] = pd.to_numeric(lr_p1_view['param_C'])
display(lr_p1_view.sort_values(['sweep_param', 'param_C']))
display(best_trial(lr_p1_results).to_frame().T[['sweep_param', 'param_C', 'mean_test_score', 'std_test_score', 'mean_train_score']])

=== LR Phase 1 ===
anchor_params={'C': 1.0}
total_candidates=11, total_fits=55
Scanning C: [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100]
Fitting 5 folds for each of 11 candidates, totalling 55 fits


Best C: value=0.1, cv_auc=0.748406, elapsed=38.9s
Saved 11 trials to hpsearch_lr_p1.csv in 38.9s


,sweep_param,param_C,mean_test_score,std_test_score,mean_train_score,std_train_score,rank_test_score
10,C,0.001,0.737891,0.005258,0.740464,0.001130,11
9,C,0.003,0.745431,0.004900,0.748280,0.001079,10
8,C,0.010,0.748106,0.004540,0.751084,0.001069,9
1,C,0.030,0.748404,0.004391,0.751428,0.001069,2
0,C,0.100,0.748406,0.004346,0.751424,0.001075,1
2,C,0.300,0.748390,0.004326,0.751405,0.001071,3
3,C,1.000,0.748384,0.004318,0.751398,0.001074,4
4,C,3.000,0.748376,0.004330,0.751395,0.001074,5
5,C,10.000,0.748374,0.004326,0.751396,0.001075,6
7,C,30.000,0.748369,0.004324,0.751394,0.001076,8


,sweep_param,param_C,mean_test_score,std_test_score,mean_train_score
0,C,0.1,0.748406,0.004346,0.751424


In [55]:
rf_p1_results = run_or_load_phase1_sweeps(
    'RF',
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=MODEL_FIT_N_JOBS),
    phase1_anchor_params['RF'],
    phase1_sweep_spaces['RF'],
    X_train_raw,
    y_train,
    phase1_csv_paths['RF'],
    rerun=PHASE1_RERUN,
)

rf_p1_view = rf_p1_results[
    [
        'sweep_param',
        'param_n_estimators',
        'param_max_depth',
        'param_min_samples_leaf',
        'param_max_features',
        'mean_test_score',
        'std_test_score',
        'mean_train_score',
        'rank_test_score',
    ]
] .copy()
display(rf_p1_view.sort_values(['sweep_param', 'rank_test_score', 'mean_test_score'], ascending=[True, True, False]))
display(best_trial_per_sweep(rf_p1_results)[
    ['sweep_param', 'param_n_estimators', 'param_max_depth', 'param_min_samples_leaf', 'param_max_features', 'mean_test_score', 'std_test_score', 'mean_train_score']
])

=== RF Phase 1 ===
anchor_params={'n_estimators': 500, 'max_depth': 28, 'min_samples_leaf': 4, 'max_features': 0.08}
output_csv=hpsearch_rf_p1_round3.csv
total_candidates=6, total_fits=30
Scanning max_features: [0.04, 0.06, 0.08, 0.1, 0.12, 'sqrt']
Fitting 5 folds for each of 6 candidates, totalling 30 fits


Best max_features: value=0.04, cv_auc=0.728822, elapsed=1269.5s
Saved 6 trials to hpsearch_rf_p1_round3.csv in 1269.5s


,sweep_param,param_n_estimators,param_max_depth,param_min_samples_leaf,param_max_features,mean_test_score,std_test_score,mean_train_score,rank_test_score
0,max_features,500,28,4,0.04,0.728822,0.005168,0.999999,1
1,max_features,500,28,4,0.06,0.728429,0.005246,1.000000,2
2,max_features,500,28,4,0.08,0.727869,0.004943,0.999999,3
3,max_features,500,28,4,sqrt,0.727751,0.004979,0.999999,4
4,max_features,500,28,4,0.1,0.726719,0.004834,0.999999,5
5,max_features,500,28,4,0.12,0.726522,0.005657,0.999999,6


,sweep_param,param_n_estimators,param_max_depth,param_min_samples_leaf,param_max_features,mean_test_score,std_test_score,mean_train_score
0,max_features,500,28,4,0.04,0.728822,0.005168,0.999999


In [ ]:
xgb_p1_results = run_or_load_phase1_sweeps(
    'XGB',
    XGBClassifier(
        random_state=RANDOM_STATE,
        n_jobs=MODEL_FIT_N_JOBS,
        verbosity=0,
        eval_metric='logloss',
    ),
    phase1_anchor_params['XGB'],
    phase1_sweep_spaces['XGB'],
    X_train_raw,
    y_train,
    phase1_csv_paths['XGB'],
    rerun=PHASE1_RERUN,
)

xgb_p1_view = xgb_p1_results[
    [
        'sweep_param',
        'param_learning_rate',
        'param_max_depth',
        'param_n_estimators',
        'param_subsample',
        'param_colsample_bytree',
        'mean_test_score',
        'std_test_score',
        'mean_train_score',
        'rank_test_score',
    ]
] .copy()
display(xgb_p1_view.sort_values(['sweep_param', 'rank_test_score', 'mean_test_score'], ascending=[True, True, False]))
display(best_trial_per_sweep(xgb_p1_results)[
    ['sweep_param', 'param_learning_rate', 'param_max_depth', 'param_n_estimators', 'param_subsample', 'param_colsample_bytree', 'mean_test_score', 'std_test_score', 'mean_train_score']
])

=== XGB Phase 1 ===
anchor_params={'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 300, 'subsample': 0.9, 'colsample_bytree': 1.0}
output_csv=hpsearch_xgb_p1.csv
total_candidates=27, total_fits=135
Scanning learning_rate: [0.01, 0.03, 0.05, 0.1, 0.2, 0.3]
Fitting 5 folds for each of 6 candidates, totalling 30 fits


## Phase 1 Visualizations

These figures summarize one-dimensional sweep results only. Use them to decide whether each parameter should be expanded, narrowed, or rescanned with a finer grid in the next Phase 1 round before defining the Phase 2 joint search space.

In [32]:
phase1_plot_paths = {
    'lr_curve': results_dir / 'hpsearch_lr_curve_p1.png',
    'rf_sensitivity': results_dir / 'hpsearch_rf_sensitivity_p1.png',
    'xgb_sensitivity': results_dir / 'hpsearch_xgb_sensitivity_p1.png',
}


def normalize_param_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.lower() == 'none':
            return None
        try:
            numeric = float(stripped)
            return int(numeric) if numeric.is_integer() else numeric
        except ValueError:
            return stripped
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        return int(value) if float(value).is_integer() else float(value)
    return value


def normalize_param_series(series):
    return series.map(normalize_param_value)


def format_param_value(value):
    value = normalize_param_value(value)
    if value is None:
        return 'None'
    return str(value)


def ordered_param_summary(results_df, param_col, order):
    working = results_df.copy()
    if param_col.startswith('param_') and 'sweep_param' in working.columns:
        target_param = param_col.replace('param_', '')
        working = working[working['sweep_param'] == target_param].copy()
    working['_param'] = normalize_param_series(working[param_col])
    normalized_order = [normalize_param_value(value) for value in order]
    summary = (
        working.groupby('_param', dropna=False)['mean_test_score']
        .agg(['mean', 'std'])
        .reindex(normalized_order)
        .reset_index()
        .rename(columns={'_param': 'param_value'})
    )
    summary['label'] = summary['param_value'].map(format_param_value)
    return summary


def save_lr_curve(results_df, out_path, title):
    plot_df = filter_sweep_results(results_df, 'C')[
        ['param_C', 'mean_test_score', 'std_test_score', 'mean_train_score', 'std_train_score']
    ].copy()
    plot_df['param_C'] = pd.to_numeric(plot_df['param_C'])
    plot_df = plot_df.sort_values('param_C')

    fig, ax = plt.subplots(figsize=(5, 3.5))
    ax.errorbar(
        np.log10(plot_df['param_C']),
        plot_df['mean_test_score'],
        yerr=plot_df['std_test_score'],
        marker='o',
        label='CV AUC',
    )
    ax.errorbar(
        np.log10(plot_df['param_C']),
        plot_df['mean_train_score'],
        yerr=plot_df['std_train_score'],
        marker='s',
        alpha=0.5,
        label='Train AUC',
    )
    ax.set_xlabel(r'$\log_{10} C$')
    ax.set_ylabel('ROC-AUC')
    ax.set_title(title)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)


def save_sensitivity_grid(results_df, param_specs, out_path, title, figsize):
    fig, axes = plt.subplots(1, len(param_specs), figsize=figsize, sharey=True)
    if len(param_specs) == 1:
        axes = [axes]

    for ax, (param_name, axis_label, order) in zip(axes, param_specs):
        summary = ordered_param_summary(results_df, f'param_{param_name}', order)
        x_pos = np.arange(len(summary))
        ax.errorbar(
            x_pos,
            summary['mean'],
            yerr=summary['std'].fillna(0),
            marker='o',
        )
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            summary['label'],
            rotation=30 if len(summary) > 3 else 0,
            ha='right',
        )
        ax.set_xlabel(axis_label)
        ax.set_title(axis_label)
        ax.grid(True, axis='y', alpha=0.3)

    axes[0].set_ylabel('CV ROC-AUC')
    fig.suptitle(title, y=1.04)
    fig.tight_layout()
    fig.savefig(out_path, dpi=FIGURE_DPI, bbox_inches='tight')
    plt.close(fig)

In [49]:
available_phase1_results = {}
for model_name, variable_name in [('LR', 'lr_p1_results'), ('RF', 'rf_p1_results'), ('XGB', 'xgb_p1_results')]:
    results_df = None
    expected_anchor = str(phase1_anchor_params[model_name])
    expected_sweeps = set(phase1_sweep_spaces[model_name])

    if phase1_csv_paths[model_name].exists():
        results_df = pd.read_csv(phase1_csv_paths[model_name])
    elif variable_name in globals():
        candidate_df = globals()[variable_name]
        if isinstance(candidate_df, pd.DataFrame) and {'anchor_params', 'sweep_param'}.issubset(candidate_df.columns):
            observed_anchor_series = candidate_df['anchor_params'].dropna()
            observed_anchor = str(observed_anchor_series.iloc[0]) if not observed_anchor_series.empty else None
            observed_sweeps = set(candidate_df['sweep_param'].dropna().unique())
            if observed_anchor == expected_anchor and expected_sweeps.issubset(observed_sweeps):
                results_df = candidate_df
            else:
                print(f'Skipping stale {model_name} in-memory Phase 1 results; rerun the current search cell first.')
        else:
            results_df = candidate_df

    if results_df is not None:
        available_phase1_results[model_name] = results_df

if 'LR' in available_phase1_results:
    save_lr_curve(available_phase1_results['LR'], phase1_plot_paths['lr_curve'], 'Logistic Regression Phase 1')
else:
    print('Skipping LR curve: no Phase 1 LR results found yet.')

if 'RF' in available_phase1_results:
    save_sensitivity_grid(
        available_phase1_results['RF'],
        [
            ('n_estimators', 'n_estimators', phase1_sweep_spaces['RF']['n_estimators']),
            ('max_features', 'max_features', phase1_sweep_spaces['RF']['max_features']),
        ],
        phase1_plot_paths['rf_sensitivity'],
        'Random Forest Phase 1 current-round sweeps',
        (7.2, 3.2),
    )
else:
    print('Skipping RF plots: no current-round Phase 1 RF results found yet.')

if 'XGB' in available_phase1_results:
    save_sensitivity_grid(
        available_phase1_results['XGB'],
        [
            ('learning_rate', 'learning_rate', phase1_sweep_spaces['XGB']['learning_rate']),
            ('max_depth', 'max_depth', phase1_sweep_spaces['XGB']['max_depth']),
            ('n_estimators', 'n_estimators', phase1_sweep_spaces['XGB']['n_estimators']),
            ('subsample', 'subsample', phase1_sweep_spaces['XGB']['subsample']),
            ('colsample_bytree', 'colsample_bytree', phase1_sweep_spaces['XGB']['colsample_bytree']),
        ],
        phase1_plot_paths['xgb_sensitivity'],
        'XGBoost Phase 1 one-dimensional sweeps',
        (16, 3.2),
    )
else:
    print('Skipping XGB plots: no Phase 1 XGB results found yet.')

phase1_plot_summary = pd.DataFrame([
    {'artifact': name, 'path': str(path.relative_to(project_root)), 'exists': path.exists()}
    for name, path in phase1_plot_paths.items()
])

display(phase1_plot_summary)

Skipping stale RF in-memory Phase 1 results; rerun the current search cell first.
Skipping RF plots: no current-round Phase 1 RF results found yet.


,artifact,path,exists
0,lr_curve,results/hpsearch_lr_curve_p1.png,True
1,rf_sensitivity,results/hpsearch_rf_sensitivity_p1.png,False
2,xgb_sensitivity,results/hpsearch_xgb_sensitivity_p1.png,True


## Phase 2 Scaffold

Do not run Phase 2 until Phase 1 has converged to compact per-parameter ranges. Phase 2 should consume the final ranges chosen after one or more Phase 1 sweep rounds and then perform the joint search only inside that narrowed space.

In [ ]:
lr_p2_grid = {}
rf_p2_dist = {
    # Discrete RF Phase 2 candidate lists, with max_features centered on the round-3 winner 0.04.
    'n_estimators': [300, 500, 700],
    'max_depth': [16, 20, 28, None],
    'min_samples_leaf': [4, 6, 8, 12],
    'max_features': [0.02, 0.04, 0.06],
}
xgb_p2_dist = {}

phase2_output_paths = {
    'lr_csv': results_dir / 'hpsearch_lr_p2.csv',
    'rf_csv': results_dir / 'hpsearch_rf_p2.csv',
    'xgb_csv': results_dir / 'hpsearch_xgb_p2.csv',
    'best_params': results_dir / 'best_params.csv',
    'main_clean': results_dir / 'main_clean.csv',
}

phase2_guidance = pd.DataFrame([
    {
        'next_action': 'Repeat Phase 1 if needed',
        'what_to_check': 'If the best point is still at a range boundary or the curve is too coarse, run another one-dimensional sweep with a wider or finer grid.',
    },
    {
        'next_action': 'Finalize Phase 2 ranges',
        'what_to_check': 'RF now uses discrete enumerated values in rf_p2_dist; fill lr_p2_grid and xgb_p2_dist after their Phase 1 sweeps stabilize.',
    },
    {
        'next_action': 'Run joint search',
        'what_to_check': 'Phase 2 should search parameter interactions only inside those finalized ranges.',
    },
])

display(phase2_guidance)

,next_action,what_to_check
0,Repeat Phase 1 if needed,If the best point is still at a range boundary...
1,Finalize Phase 2 ranges,RF now uses discrete enumerated values in rf_p...
2,Run joint search,Phase 2 should search parameter interactions o...


In [34]:
def build_phase2_searches():
    phase2_spaces = {
        'LR': lr_p2_grid,
        'RF': rf_p2_dist,
        'XGB': xgb_p2_dist,
    }
    missing = [model_name for model_name, space in phase2_spaces.items() if not space]
    if missing:
        print('Phase 2 is scaffolded only.')
        print('Fill the joint search spaces after Phase 1 ranges are finalized.')
        print(f'Missing or empty search spaces: {", ".join(missing)}')
        return None

    return {
        'LR': GridSearchCV(
            build_model_pipeline(LogisticRegression(solver='lbfgs', max_iter=1000)),
            param_grid=prefix_model_params(lr_p2_grid),
            cv=cv,
            scoring='roc_auc',
            n_jobs=SEARCH_N_JOBS,
            pre_dispatch=SEARCH_PRE_DISPATCH,
            return_train_score=True,
            verbose=SEARCH_VERBOSE,
        ),
        'RF': RandomizedSearchCV(
            build_model_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=MODEL_FIT_N_JOBS)),
            param_distributions=prefix_model_params(rf_p2_dist),
            n_iter=50,
            cv=cv,
            scoring='roc_auc',
            n_jobs=SEARCH_N_JOBS,
            pre_dispatch=SEARCH_PRE_DISPATCH,
            return_train_score=True,
            random_state=RANDOM_STATE,
            verbose=SEARCH_VERBOSE,
        ),
        'XGB': RandomizedSearchCV(
            build_model_pipeline(
                XGBClassifier(
                    random_state=RANDOM_STATE,
                    n_jobs=MODEL_FIT_N_JOBS,
                    verbosity=0,
                    eval_metric='logloss',
                )
            ),
            param_distributions=prefix_model_params(xgb_p2_dist),
            n_iter=50,
            cv=cv,
            scoring='roc_auc',
            n_jobs=SEARCH_N_JOBS,
            pre_dispatch=SEARCH_PRE_DISPATCH,
            return_train_score=True,
            random_state=RANDOM_STATE,
            verbose=SEARCH_VERBOSE,
        ),
    }


phase2_searches = build_phase2_searches()
phase2_searches

Phase 2 is scaffolded only.
Fill the joint search spaces after Phase 1 ranges are finalized.
Missing or empty search spaces: LR, RF, XGB


## Artifact Summary

Use this table to confirm which Phase 1 deliverables already exist and which Phase 2 outputs are intentionally pending.

In [ ]:
expected_artifacts = {
    'phase1_csv': list(phase1_csv_paths.values()) + [
        results_dir / 'phase1_refinement_summary.csv',
        results_dir / 'phase2_range_suggestions.csv',
    ],
    'phase1_png': list(phase1_plot_paths.values()),
    'phase2_outputs': list(phase2_output_paths.values()),
}

artifact_rows = []
for artifact_group, paths in expected_artifacts.items():
    for path in paths:
        artifact_rows.append({
            'artifact_group': artifact_group,
            'file_name': path.name,
            'relative_path': str(path.relative_to(project_root)),
            'exists': path.exists(),
        })

artifact_summary = pd.DataFrame(artifact_rows)
display(artifact_summary)

,artifact_group,file_name,relative_path,exists
0,phase1_csv,hpsearch_lr_p1.csv,results/hpsearch_lr_p1.csv,True
1,phase1_csv,hpsearch_rf_p1.csv,results/hpsearch_rf_p1.csv,False
2,phase1_csv,hpsearch_xgb_p1.csv,results/hpsearch_xgb_p1.csv,False
3,phase1_csv,phase1_signals.csv,results/phase1_signals.csv,True
4,phase1_png,hpsearch_lr_curve_p1.png,results/hpsearch_lr_curve_p1.png,True
5,phase1_png,hpsearch_rf_heatmap_p1.png,results/hpsearch_rf_heatmap_p1.png,False
6,phase1_png,hpsearch_xgb_heatmap_p1.png,results/hpsearch_xgb_heatmap_p1.png,False
7,phase1_png,hpsearch_rf_sensitivity_p1.png,results/hpsearch_rf_sensitivity_p1.png,False
8,phase1_png,hpsearch_xgb_sensitivity_p1.png,results/hpsearch_xgb_sensitivity_p1.png,False
9,phase2_outputs,hpsearch_lr_p2.csv,results/hpsearch_lr_p2.csv,False
